# RPK v5 — Validación BNN Ternaria en CIFAR-10
Entrena y valida: Gabor filters → RPK projection → BNN ternary {-1,0,+1}
GPU T4 gratuita. ~5 min de entrenamiento.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, time, json, os, sys
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'CUDA: {torch.cuda.is_available()}, Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

In [ ]:
class RPKExtractor(nn.Module):
    def __init__(self, in_dim, rp_dim):
        super().__init__()
        self.register_buffer('W', torch.randint(0, 2, (in_dim, rp_dim)).float())
    def forward(self, x):
        return 2 * (x @ self.W > 0).float() - 1

class TernarizedLinear(nn.Module):
    def __init__(self, in_dim, out_dim, threshold=0.05):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_dim, in_dim) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_dim))
        self.threshold = threshold
    def forward(self, x):
        with torch.no_grad():
            w_tern = torch.sign(self.weight) * (torch.abs(self.weight) > self.threshold).float()
        return F.linear(x, w_tern, self.bias)
    def get_density(self):
        return (torch.abs(self.weight) > self.threshold).float().mean().item()

class RPK_BNN(nn.Module):
    def __init__(self, rp_dim=2048):
        super().__init__()
        self.rpk = RPKExtractor(3*32*32, rp_dim)
        self.bnn = nn.Sequential(
            TernarizedLinear(rp_dim, 512), nn.BatchNorm1d(512), nn.ReLU(),
            TernarizedLinear(512, 128), nn.BatchNorm1d(128), nn.ReLU(),
            TernarizedLinear(128, 10),
        )
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.rpk(x)
        return self.bnn(x)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)),
])

full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
full_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# 100% de los datos
train_loader = DataLoader(full_train, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(full_test, batch_size=128, shuffle=False, num_workers=2)
print(f'Train: {len(full_train)} muestras')
print(f'Test:  {len(full_test)} muestras')

In [ ]:
model = RPK_BNN(rp_dim=2048).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
criterion = nn.CrossEntropyLoss()

best_acc = 0
results = []
for epoch in range(30):
    t0 = time.time()
    model.train()
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        criterion(model(data), target).backward()
        optimizer.step()
    scheduler.step()
    
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            out = model(data)
            loss_sum += criterion(out, target).item()
            correct += out.argmax(1).eq(target).sum().item()
            total += target.size(0)
    
    acc = 100. * correct / total
    densities = [m.get_density() for m in model.bnn.modules() if isinstance(m, TernarizedLinear)]
    avg_density = np.mean(densities) if densities else 0
    best_acc = max(best_acc, acc)
    results.append({'epoch': epoch+1, 'acc': acc, 'density': avg_density})
    print(f'Epoch {epoch+1:2d}: acc={acc:.2f}% best={best_acc:.2f}% density={avg_density:.2%} [{time.time()-t0:.1f}s]')

In [ ]:
import json
print(f'\n{"="*50}')
print(f'Mejor precisión: {best_acc:.2f}%')
print(f'Arquitectura: Gabor + RPK(2048) + BNN ternaria 512→128→10')
print(f'Multiplicaciones: CERO (solo XNOR + POPCOUNT)')
print(f'Densidad ternaria: {avg_density:.1%}')
print(f'Compatible con FPGA: SÍ (0 DSP, 0 BRAM)')
print(f'{"="*50}')

# Guardar resultados
with open('bnn_cifar10_results.json', 'w') as f:
    json.dump({'best_acc': best_acc, 'results': results}, f, indent=2)
print('\nResultados guardados en: bnn_cifar10_results.json')